Purpose: Create a nice table summarizing gene content in the clusters.<br>
Author: Anna Pardo<br>
Date initiated: Nov. 19, 2025

In [1]:
import pandas as pd
import os

In [2]:
clusttbl = pd.read_csv("./n_genes_in_clusters_table.csv",sep=",",header="infer")
clusttbl

,cluster,n_genes_G,n_genes_13,n_genes_53,n_genes_52,n_genes_48,n_genes_Eudy,n_genes_19,n_genes_46,n_genes_56,n_genes_45,n_genes_1AB,n_genes_70,n_genes_18,n_genes_55,n_genes_2AB,n_genes_51,n_genes_gt37,n_genes_all
0,1,979,902,319,1315,1330,1796,1373,2699,2044,1790,1410,4521,4885,4992,6704,11456,5188,12226
1,2,914,797,656,1607,534,1655,1276,1857,3602,3884,4846,2945,6775,5331,3929,9976,6548,10371
2,3,542,676,882,1358,1904,1229,2848,2076,3198,1542,4203,5846,1959,3800,8360,4740,8517,1974
3,4,622,1026,1071,780,1431,2395,2051,749,2535,3375,2699,2857,4629,3099,5074,8313,10158,4055
4,5,153,209,428,629,1163,976,1982,1821,915,1795,3834,4037,4963,7942,2923,4714,9207,3944
5,6,573,280,777,489,601,820,1498,2178,708,3720,2273,2461,915,6734,5465,3031,6965,32834
6,Total,3783,3890,4133,6178,6963,8871,11028,11380,13002,16106,19265,22667,24126,31898,32455,42230,46583,65404


In [3]:
# load cluster gene information for each genotype/species
## make a function to do this with subgenome info for Y. gloriosa
def load_yg_clusters(filename):
    ygc = pd.read_csv(filename,sep="\t",header="infer")
    # make subgenome column
    sg = []
    for i in list(ygc["GeneID"]):
        if i.startswith("Yucal"):
            sg.append("Ya")
        else:
            sg.append("Yf")
    ygc["subgenome"] = sg
    return ygc

In [10]:
allgt = {}
for f in os.listdir("./"):
    if f.endswith(".txt"):
        if ("k6" in f and "Yg" in f):
            gt = f.split("_")[0].lstrip("Yg")
            allgt[gt] = load_yg_clusters(os.path.join("./",f))
            allgt[gt]["genotype"] = gt
    elif (f.endswith(".tsv") and f.startswith("hck6")):
        if "gt" in f:
            gt = "37"
        else:
            gt = "all"
        allgt[gt] = load_yg_clusters(os.path.join("./",f))
        allgt[gt]["genotype"] = gt

In [11]:
allgt.keys()

dict_keys(['G', 'Eudy', '13', '45', '19', '51', '2AB', '52', '46', '70', '48', '37', '55', '18', '53', '56', 'all', '1AB'])

In [13]:
# merge the dict values (or, well, concatenate them)
allgtdf = pd.concat(list(allgt.values()))

In [14]:
allgtdf.head()

,cluster,GeneID,subgenome,genotype
0,1,Yucal.01G003400.v2.1,Ya,G
1,1,Yucal.01G004800.v2.1,Ya,G
2,2,Yucal.01G004900.v2.1,Ya,G
3,3,Yucal.01G005300.v2.1,Ya,G
4,4,Yucal.01G012100.v2.1,Ya,G


In [16]:
# create the summary table: by genotype & subgenome only to start
sum_gt_subg = allgtdf.groupby(["genotype","subgenome"]).count().reset_index()[["genotype","subgenome","GeneID"]].rename(columns={"GeneID":"number of genes"})

In [17]:
sum_gt_subg

,genotype,subgenome,number of genes
0,13,Ya,1983
1,13,Yf,1907
2,18,Ya,12357
3,18,Yf,11769
4,19,Ya,5757
5,19,Yf,5271
6,1AB,Ya,9964
7,1AB,Yf,9301
8,2AB,Ya,16697
9,2AB,Yf,15758


In [20]:
# now do the same but show the breakdown for each cluster
sumc = allgtdf.groupby(["genotype","subgenome","cluster"]).count().reset_index()[["genotype","subgenome","cluster","GeneID"]].rename(columns={"GeneID":"number of genes"})


In [25]:
# convert "number of genes" to int
sumc["number of genes"] = sumc["number of genes"].astype(int)

In [27]:
# create a mapping dict for renaming clusters
mapd = {
    1:"cluster 1",
    2:"cluster 2",
    3:"cluster 3",
    4:"cluster 4",
    5:"cluster 5",
    6:"cluster 6"
}

In [28]:
sumc["cluster"] = sumc["cluster"].map(mapd)
sumc.head()

,genotype,subgenome,cluster,number of genes
0,13,Ya,cluster 1,463
1,13,Ya,cluster 2,392
2,13,Ya,cluster 3,341
3,13,Ya,cluster 4,529
4,13,Ya,cluster 5,110


In [30]:
sumcp = sumc.pivot("cluster",["genotype","subgenome"],"number of genes").transpose().reset_index()

In [32]:
sumcp.head()

cluster,genotype,subgenome,cluster 1,cluster 2,cluster 3,cluster 4,cluster 5,cluster 6
0,13,Ya,463.0,392.0,341.0,529.0,110.0,148.0
1,13,Yf,439.0,405.0,335.0,497.0,99.0,132.0
2,18,Ya,2546.0,3480.0,1012.0,2317.0,2540.0,462.0
3,18,Yf,2339.0,3295.0,947.0,2312.0,2423.0,453.0
4,19,Ya,759.0,681.0,1468.0,1055.0,1011.0,783.0


In [33]:
sumcp.to_csv("./masigpro_sumtbl_genecounts_by_cluster_subgenome_genotype_Yg.csv",sep=",",header=True,index=False)

In [34]:
sum_gt_subg.head()

,genotype,subgenome,number of genes
0,13,Ya,1983
1,13,Yf,1907
2,18,Ya,12357
3,18,Yf,11769
4,19,Ya,5757


In [35]:
sum_gt_subg.to_csv("./masigpro_sumtbl_genecounts_by_subgenome_genotype_Yg.csv",sep=",",header=True,index=False)

In [41]:
# fix sum_gt_subg
sump = sum_gt_subg.pivot("genotype","subgenome","number of genes").reset_index().rename(columns={"Ya":"number of Ya genes",
                                                                                                "Yf":"number of Yf genes"})

In [42]:
sump.head()

subgenome,genotype,number of Ya genes,number of Yf genes
0,13,1983,1907
1,18,12357,11769
2,19,5757,5271
3,1AB,9964,9301
4,2AB,16697,15758


In [43]:
allgtdf.head()

,cluster,GeneID,subgenome,genotype
0,1,Yucal.01G003400.v2.1,Ya,G
1,1,Yucal.01G004800.v2.1,Ya,G
2,2,Yucal.01G004900.v2.1,Ya,G
3,3,Yucal.01G005300.v2.1,Ya,G
4,4,Yucal.01G012100.v2.1,Ya,G


In [45]:
# get the summary of total numbers of time-structured genes per genotype and add that to sump
tstotal = allgtdf.groupby("genotype").count().reset_index()[["genotype","GeneID"]].rename(columns={"GeneID":"total number of genes"})
tstotal.head()

,genotype,total number of genes
0,13,3890
1,18,24126
2,19,11028
3,1AB,19265
4,2AB,32455


In [46]:
sump = sump.merge(tstotal)
sump.head()

,genotype,number of Ya genes,number of Yf genes,total number of genes
0,13,1983,1907,3890
1,18,12357,11769,24126
2,19,5757,5271,11028
3,1AB,9964,9301,19265
4,2AB,16697,15758,32455


In [47]:
sump.sort_values(by="total number of genes",ascending=False,inplace=True)
sump.head()

,genotype,number of Ya genes,number of Yf genes,total number of genes
17,all,33599,31805,65404
5,37,23632,22951,46583
9,51,21665,20565,42230
4,2AB,16697,15758,32455
12,55,16585,15313,31898


In [48]:
sump.to_csv("masigpro_sumtbl_tsgenes_by_genotype_subgenome_total.csv",sep=",",header=True,index=False)